In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

from tyche import DependenceEstimator, DependenceVolumeConfig, VolumeConfig, VolumeEstimator

import random
from datasets import Dataset
import string

def generate_token(length=50):
    return ''.join(random.choices(string.ascii_lowercase, k=length))

# Option 1: 50 separate examples (each with a single random token)
ref_data = {"text": [generate_token() for _ in range(50)]}
ref_dataset = Dataset.from_dict(ref_data)

# Load any CausalLM model, tokenizer, and dataset
model = AutoModelForCausalLM.from_pretrained("EleutherAI/pythia-14m")
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/pythia-14m")
tokenizer.pad_token_id = 1  # pythia-specific
tokenizer.eos_token_id = 0  # pythia-specific
dataset = load_dataset("EleutherAI/lambada_openai", name="en", split="test", trust_remote_code=True)
# dataset2 = load_dataset("EleutherAI/lambada_openai", name="de", split="test", trust_remote_code=True)
dataset2 = load_dataset("EleutherAI/quirky_hemisphere_alice", split="test")


# Configure the estimator
cfg = DependenceVolumeConfig(model=model, 
                   tokenizer=tokenizer, 
                   dataset=dataset, 
                   dataset2=dataset2,
                   dataset_ref=ref_dataset,
                   text_key="text",  # must match dataset field
                   text_key2="statement",  # must match dataset2 field
                   n_samples=50,  # number of MC samples
                   cutoff=1e-2,  # KL-divergence cutoff (nats)
                   max_seq_len=2048,  # max sequence length for tokenizer or chunk_and_tokenize
                   val_size=10,  # number of dataset sequences to use. default (None) uses all.
                   cache_mode=None,  # see below
                   chunking=False,  # whether to use chunk_and_tokenize
                   scale_ref=0.8
                   )
estimator = DependenceEstimator.from_config(cfg)

# Run the estimator
# result = estimator.run()

/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The `GPTNeoXSdpaAttention` class is deprecated in favor of simply modifying the `config._attn_implementation`attribute of the `GPTNeoXAttention` class! It will be removed in v4.48


In [4]:
(result.estimates['joint'] - result.estimates['ref']).mean(), (result.estimates['marginal1'] + result.estimates['marginal2'] - 2 * result.estimates['ref']).mean()

(tensor(-819778.1875, device='cuda:0'), tensor(-1179586.8750, device='cuda:0'))

In [10]:
(result.estimates['joint'] - result.estimates['ref']).mean(), (result.estimates['marginal1'] + result.estimates['marginal2'] - 2 * result.estimates['ref']).mean()

(tensor(-125483.3594, device='cuda:0'), tensor(-125483.5156, device='cuda:0'))

In [ ]:
%pdb on

cfg = VolumeConfig(model=model, 
                   tokenizer=tokenizer, 
                   dataset=dataset, 
                   dataset2=dataset2,
                   text_key="text",  # must match dataset field
                   text_key2="statement",  # must match dataset2 field
                   n_samples=50,  # number of MC samples
                   cutoff=1e-2,  # KL-divergence cutoff (nats)
                   max_seq_len=2048,  # max sequence length for tokenizer or chunk_and_tokenize
                   val_size=10,  # number of dataset sequences to use. default (None) uses all.
                   cache_mode=None,  # see below
                   chunking=False,  # whether to use chunk_and_tokenize
                   scale_ref=0.8,
                   checkpoint_step=143000,
                   preconditioner_type="adam",
                   model_type="pythia",
                   cache_mode="cpu"",
                   )
estimator = VolumeEstimator.from_config(cfg)
estimator.run()

Automatic pdb calling has been turned ON


Matching parameters in group 0
Matching parameters in group 1
Estimating joint volume


  0%|          | 0/50 [00:00<?, ?it/s]


TypeError: kl_div(): argument 'target' (position 2) must be Tensor, not NoneType

> /opt/conda/lib/python3.11/site-packages/torch/nn/functional.py(3396)kl_div()
   3394             reduction_enum = _Reduction.get_enum(reduction)
   3395 
-> 3396     reduced = torch.kl_div(input, target, reduction_enum, log_target=log_target)
   3397 
   3398     if reduction == "batchmean" and input.dim() != 0:



In [9]:
result.estimates['joint'], result.estimates['marginal1'], result.estimates['marginal2'], result.estimates['ref']

(tensor([-1.0854e+08, -1.0957e+08, -1.0814e+08, -1.0884e+08, -1.1965e+08,
         -1.0864e+08, -1.0874e+08, -1.0889e+08, -1.0899e+08, -1.0834e+08,
         -1.0864e+08, -1.0864e+08, -1.0864e+08, -1.0904e+08, -1.0946e+08,
         -1.0915e+08, -1.0874e+08, -1.0879e+08, -1.0748e+08, -1.0931e+08,
         -1.0979e+08, -1.0904e+08, -1.0968e+08, -1.0809e+08, -1.0915e+08,
         -1.0844e+08, -1.0844e+08, -1.0869e+08, -1.0419e+08, -1.0879e+08,
         -1.0973e+08, -1.0894e+08, -1.0968e+08, -1.0844e+08, -1.0844e+08,
         -1.0968e+08, -1.0884e+08, -1.0864e+08, -1.0819e+08, -1.0899e+08],
        device='cuda:0'),
 tensor([-1.0854e+08, -1.0834e+08, -1.0739e+08, -1.0809e+08, -1.0748e+08,
         -1.0819e+08, -1.0884e+08, -1.0814e+08, -1.0874e+08, -1.0748e+08,
         -1.0844e+08, -1.0748e+08, -1.0839e+08, -1.0730e+08, -1.0819e+08,
         -1.0915e+08, -1.0879e+08, -1.0805e+08, -1.0795e+08, -1.0864e+08,
         -1.0979e+08, -1.0767e+08, -1.0915e+08, -1.0809e+08, -1.0904e+08,
         -1